# 01 · Near-duplicate removal
**Paper:** Step 1 · Fig. 2A

Many Quilt-1M images are video frames grabbed while the presenter slowly pans across a slide,
so neighbouring frames are almost identical. We:

1. compute a **perceptual hash (pHash)** for every image,
2. index the hashes in a **BK-tree** for fast similarity search,
3. treat two images as near-duplicates when their pHashes differ by **≤ 5 bits** (Hamming distance).

Images are visited in sorted filename order. The first image of each near-duplicate group is
kept and the rest are excluded from the pool from which the held-out evaluation sample is
drawn (notebook 04).

**Output** → `outputs/phashes.csv`, `outputs/near_duplicate_exclusions.csv`

In [ ]:
# --- setup: make the `requilt` package importable and load the shared config ---
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))   # repo root (not needed after `pip install -e .`)

from requilt import load_config

cfg = load_config()      # configs/config.yaml
P = cfg.paths            # all paths, already resolved to absolute Paths

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image

from requilt.dedup import compute_phashes, find_near_duplicates

## 1 · Hash every image
Hashing all 653,213 images takes a while, so the hashes are cached in `outputs/phashes.csv`.
Delete that file to recompute them.

In [ ]:
hash_cache = P.output_dir / "phashes.csv"

if hash_cache.exists():
    cached = pd.read_csv(hash_cache, dtype={"phash": str})
    hashes = {name: int(h, 16) for name, h in zip(cached["Filename"], cached["phash"])}
    print(f"loaded {len(hashes):,} cached hashes")
else:
    image_paths = sorted(P.quilt_images_dir.iterdir())
    hashes = compute_phashes(image_paths,
                             hash_size=cfg.dedup.hash_size,
                             num_workers=cfg.dedup.num_workers)
    pd.DataFrame({"Filename": list(hashes),
                  "phash": [f"{h:016x}" for h in hashes.values()]}).to_csv(hash_cache, index=False)
    print(f"hashed {len(hashes):,} images -> {hash_cache}")

## 2 · Find near-duplicates with the BK-tree

In [ ]:
removed = pd.DataFrame(find_near_duplicates(hashes, max_distance=cfg.dedup.max_hamming_distance))

out_csv = P.output_dir / "near_duplicate_exclusions.csv"
removed.to_csv(out_csv, index=False)

print(f"images hashed          : {len(hashes):,}")
print(f"near-duplicates removed: {len(removed):,}  ({len(removed) / len(hashes):.1%})")
print(f"images kept            : {len(hashes) - len(removed):,}")
print(f"saved -> {out_csv}")